In [ ]:
%%file consumer_anomaly.py
from kafka import KafkaConsumer
import json
from datetime import datetime

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='grupa_anomalie',
    value_deserializer=lambda v: json.loads(v.decode('utf-8'))
)

historia_user = {}

for message in consumer:
    tx = message.value
    user = tx['user_id']
    
    # zamiana stringa na czas
    czas_obecny = datetime.fromisoformat(tx['timestamp'])
    
    if user not in historia_user:
        historia_user[user] = []
        
    historia_user[user].append(czas_obecny)
    
    # czyszczenie starych transakcji
    nowa_lista = []
    for t in historia_user[user]:
        roznica_sekundy = (czas_obecny - t).total_seconds()
        if roznica_sekundy <= 60:
            nowa_lista.append(t)
            
    historia_user[user] = nowa_lista
    
    if len(historia_user[user]) > 3:
        print(f"ANOMALIA! User {user} mial {len(historia_user[user])} transakcji w ciagu 60s")